# Konux Case Study – End-to-End Solution

This notebook solves the Konux case study **step by step**, following the requested workflow:

1. Raw signal vs clean signal visualization
2. Feature engineering
3. Outlier detection (before classification)
4. Outlier removal
5. Train-type classification (clustering)
6. Relative train speed analysis

All steps are aligned with the official case-study description.

## 1. Imports and Global Parameters

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import scipy.signal as sig
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

FS = 2000.0  # Sampling frequency (Hz)

## 2. Load Binary Acceleration Data

In [ ]:
def read_trace(path):
    return np.fromfile(path, dtype=np.float32)

DATA_DIR = '.'
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.bin')))
print(f'Found {len(files)} files')

## 3. Signal Preprocessing

In [ ]:
def preprocess(x, fs=FS):
    x = x - np.median(x)
    sos = sig.butter(4, [5, 400], btype='bandpass', fs=fs, output='sos')
    return sig.sosfiltfilt(sos, x)

## 4. Raw Signal vs Clean Signal

In [ ]:
raw = read_trace(files[0])
clean = preprocess(raw)
t = np.arange(len(raw)) / FS

plt.figure(figsize=(12,4))
plt.plot(t, raw, alpha=0.4, label='Raw')
plt.plot(t, clean, label='Clean (Filtered)')
plt.xlabel('Time [s]'); plt.ylabel('Acceleration [g]')
plt.title('Raw vs Clean Signal')
plt.legend(); plt.show()

## 5. Train Passage Segmentation

In [ ]:
def segment_passage(x, fs=FS, win_s=0.1):
    win = int(win_s * fs)
    rms = np.sqrt(sig.convolve(x**2, np.ones(win)/win, mode='same'))
    med = np.median(rms)
    mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + 6 * mad
    idx = np.where(rms > thr)[0]
    if len(idx) == 0:
        return x
    splits = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[splits + 1]]
    ends = np.r_[idx[splits], idx[-1]]
    k = np.argmax(ends - starts)
    s, e = starts[k], ends[k]
    pad = int(0.5 * fs)
    return x[max(0,s-pad):min(len(x), e+pad)]

## 6. Feature Engineering

In [ ]:
def extract_features(x, fs=FS):
    f, Pxx = sig.welch(x, fs=fs, nperseg=min(4096, len(x)))
    env = np.abs(sig.hilbert(x))
    fe, Pe = sig.welch(env, fs=fs, nperseg=min(4096, len(env)))
    mask = (fe >= 1) & (fe <= 30)
    dom_env_freq = fe[mask][np.argmax(Pe[mask])]
    return {
        'duration_s': len(x)/fs,
        'rms': np.sqrt(np.mean(x**2)),
        'peak': np.max(np.abs(x)),
        'spec_centroid': np.sum(f*Pxx)/np.sum(Pxx),
        'env_dom_freq': dom_env_freq
    }

## 7. Build Feature Table

In [ ]:
rows = []
for fpath in files:
    raw = read_trace(fpath)
    clean = preprocess(raw)
    seg = segment_passage(clean)
    feats = extract_features(seg)
    feats['file'] = os.path.basename(fpath)
    rows.append(feats)

df = pd.DataFrame(rows)
df

## 8. Feature Heatmap

In [ ]:
features = ['duration_s','rms','peak','spec_centroid','env_dom_freq']
Xs = StandardScaler().fit_transform(df[features])

plt.figure(figsize=(8,6))
plt.imshow(Xs, aspect='auto')
plt.colorbar(label='Standardized value')
plt.yticks(range(len(df)), df['file'])
plt.xticks(range(len(features)), features, rotation=45)
plt.title('Feature Heatmap')
plt.show()

## 9. Boxplot – Outlier Detection

In [ ]:
df[features].boxplot(figsize=(8,4))
plt.title('Feature Distributions and Potential Outliers')
plt.show()

## 10. Outlier Removal (Isolation Forest)

In [ ]:
iso = IsolationForest(contamination=0.1, random_state=42)
df['outlier'] = iso.fit_predict(Xs)
df_clean = df[df['outlier'] == 1].reset_index(drop=True)

print('Removed', len(df) - len(df_clean), 'outliers')

## 11. Train Type Classification (Clustering)

In [ ]:
Xs_clean = StandardScaler().fit_transform(df_clean[features])
Z = PCA(n_components=2).fit_transform(Xs_clean)

kmeans = KMeans(n_clusters=min(4, len(df_clean)), random_state=0, n_init='auto')
df_clean['cluster'] = kmeans.fit_predict(Z)

plt.figure(figsize=(6,6))
plt.scatter(Z[:,0], Z[:,1], c=df_clean['cluster'])
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Train Type Classification (PCA)')
plt.show()

## 12. Relative Train Speed by Cluster

In [ ]:
df_sorted = df_clean.sort_values('env_dom_freq', ascending=False)

plt.figure(figsize=(8,4))
plt.barh(df_sorted['file'], df_sorted['env_dom_freq'])
plt.xlabel('Dominant Envelope Frequency [Hz]')
plt.title('Relative Train Speed')
plt.gca().invert_yaxis()
plt.show()

## 13. Save Results

In [ ]:
df_clean.to_csv('train_analysis_results_clean.csv', index=False)
print('Saved train_analysis_results_clean.csv')